# 036 — Proyecto: sistema híbrido para decisiones

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** `P(V) = 0.9·0.10 + 0.2·0.90 = 0.27`. `P(F|V) = 0.09/0.27 = 1/3 ≈ 0.333`.

**E2.** Con V: `EU(det) = −5`; `EU(cont) = (1/3)(−40) = −13.33` → **detener**. Sin V: `P(F|¬V) = 0.1·0.10/(0.1·0.10+0.8·0.90) = 0.01/0.73 ≈ 0.0137`; `EU(cont) = 0.0137·(−40) ≈ −0.55 > −5` → **continuar**. El sistema usa el sensor: la acción depende de la evidencia, como debe.

**E3.** Posterior de cruce: `−5 = p·(−40)` → `p* = 0.125`. `LR = 0.9/0.2 = 4.5`. Odds de cruce `= 0.125/0.875 = 1/7`; odds de prior `= (1/7)/4.5 = 1/31.5` → prior de equilibrio `≈ 0.031`. El prior estimado (0.10) es 3 veces mayor: habría que sobreestimar la fiabilidad de las máquinas por un factor ~3 para cambiar la acción. **Decisión robusta.**

**E4.** Fitness: utilidad media sobre N escenarios simulados (muestrear F con el prior, luego V con (TPR, FPR) del θ candidato, decidir con MEU, sumar U), con semillas comunes entre candidatos para comparar con menos ruido. A mano (esperanza exacta, prior 0.10):
- θ bajo (0.95, 0.4): `P(V)=0.455`, post_V≈0.209 → detener (−5)·0.455; post_¬V≈0.0092 → continuar (−0.367)·0.545 → **EU ≈ −2.475**.
- θ medio (0.9, 0.2): `P(V)=0.27` → detener −5·0.27 = −1.35; ¬V: post≈0.0137 → cont (−0.548)·0.73 ≈ −0.40 → **EU ≈ −1.750**.
- θ alto (0.7, 0.05): `P(V)=0.115`, post_V≈0.609 → detener −5·0.115 = −0.575; ¬V: post≈0.0339 → cont (−1.356)·0.885 ≈ −1.20 → **EU ≈ −1.775**.
Gana **θ medio por un margen pequeño (≈0.025) sobre θ alto**; θ bajo pierde claramente porque su tasa de falsas alarmas fuerza paradas innecesarias. Con Monte Carlo de N moderado, 0.025 queda dentro del error estándar típico: la respuesta honesta es "medio o alto, empate práctico; hace falta más N (o cálculo exacto, como aquí) para separarlos".


In [ ]:
result = run_lab("capstone", seed=36)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


In [ ]:
prior = 0.10
def eu_total(tpr, fpr, p=prior):
    pV = tpr*p + fpr*(1-p)
    def mejor(post):
        return max(-5.0, post*(-40.0))
    post_V = tpr*p/pV
    post_nV = (1-tpr)*p/(1-pV)
    return pV*mejor(post_V) + (1-pV)*mejor(post_nV), post_V

for nombre, (tpr, fpr) in {"bajo": (0.95, 0.4), "medio": (0.9, 0.2), "alto": (0.7, 0.05)}.items():
    eu, post = eu_total(tpr, fpr)
    print(f"theta {nombre}: EU={eu:.3f} posterior con V={post:.3f}")
LR = 0.9/0.2
odds_cruce = (0.125/0.875)/LR
print("prior de equilibrio:", round(odds_cruce/(1+odds_cruce), 3))


## Reflexión

1. En la traza del laboratorio, identifica qué campo pertenece a cada capa (creencia, decisión, optimización, trazabilidad). ¿Falta alguna capa? ¿Qué añadirías?
2. ¿Por qué "ajustar el prior para ser más precavidos" es peor diseño que "ajustar la utilidad del falso negativo"? ¿Qué se vuelve inauditable en el primer caso?
3. Propón el análisis de sensibilidad mínimo que exigirías antes de dejar que este sistema decida sin revisión humana, y el criterio cuantitativo de abstención.
